# How the Body Controller Demos work

**Want to play with the live demos?** Download [`index.html`](https://github.com/crqzyexprees/Mediapipe-demo/blob/main/index.html) and double-click it. It opens in Chrome or Edge with nothing to install. It has the finger counter, face filters with a mood detector, and full-body characters.

**This notebook shows what’s behind them,** one step at a time, in Python:

| # | Section | You’ll learn |
|---|---|---|
| 1 | What a landmark is | The raw numbers MediaPipe gives you |
| 2 | Finger counter | How “21 points” becomes “4 fingers” |
| 3 | Hands in 3D | MediaPipe estimates depth, too |
| 4 | Pose angles | Joint angles, the basis of a rep counter |
| 5 | Mood detector | Reading expressions from 52 face scores |
| 6 | **Train your own gesture classifier** | Turn landmarks into features and train a model on a free dataset |

Run the cells from top to bottom. In **Google Colab**, the photo cells can use your webcam. Anywhere else, they use Google’s sample photos.

*Made by ml@psu · Code: [github.com/crqzyexprees/Mediapipe-demo](https://github.com/crqzyexprees/Mediapipe-demo)*

## 0. Setup

Installs MediaPipe (about 30 seconds in Colab). If Colab asks you to restart the session, restart and run from this cell again.

In [ ]:
%pip install -q mediapipe==1.0.1
try:
    import sklearn
except ImportError:
    %pip install -q scikit-learn

In [ ]:
import os, sys, time, math, urllib.request, zipfile, random
import numpy as np
import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

IN_COLAB = "google.colab" in sys.modules
USE_WEBCAM = True        # Colab only: take photos with your webcam. Set False to always use sample photos.

MODELS = "https://storage.googleapis.com/mediapipe-models/"
HAND_MODEL = MODELS + "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
FACE_MODEL = MODELS + "face_landmarker/face_landmarker/float16/1/face_landmarker.task"
POSE_MODEL = MODELS + "pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task"
SAMPLES = {
    "hands": "https://storage.googleapis.com/mediapipe-tasks/hand_landmarker/woman_hands.jpg",
    "face": "https://storage.googleapis.com/mediapipe-assets/business-person.png",
    "pose": "https://storage.googleapis.com/mediapipe-assets/pose.jpg",
}

def fetch(url, path, tries=3):
    """Download a file once, with retries."""
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    for i in range(tries):
        try:
            urllib.request.urlretrieve(url, path)
            return path
        except Exception as e:
            print(f"Download failed ({e}), retrying…")
            time.sleep(1 + i)
    raise RuntimeError(f"Couldn't download {url}. Check your internet connection.")

def model(url):
    return fetch(url, url.rsplit("/", 1)[-1])

def take_photo(filename="webcam.jpg"):
    """Colab only: shows your webcam with a 'Take photo' button and saves the picture."""
    from base64 import b64decode
    from IPython.display import Javascript, display
    from google.colab.output import eval_js
    display(Javascript('''
      async function takePhoto() {
        const box = document.createElement("div");
        const video = document.createElement("video");
        const button = document.createElement("button");
        button.textContent = "Take photo";
        button.style.cssText = "font-size:16px;padding:6px 14px;margin-top:6px";
        video.style.cssText = "display:block;max-width:100%";
        box.append(video, button);
        document.body.appendChild(box);
        const stream = await navigator.mediaDevices.getUserMedia({ video: { width: 1280, height: 720 } });
        video.srcObject = stream;
        await video.play();
        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        await new Promise((resolve) => (button.onclick = resolve));
        const canvas = document.createElement("canvas");
        canvas.width = video.videoWidth; canvas.height = video.videoHeight;
        canvas.getContext("2d").drawImage(video, 0, 0);
        stream.getTracks().forEach((t) => t.stop());
        box.remove();
        return canvas.toDataURL("image/jpeg", 0.92);
      }'''))
    data = eval_js("takePhoto()")
    with open(filename, "wb") as f:
        f.write(b64decode(data.split(",")[1]))
    return filename

def photo(kind):
    """Webcam photo in Colab, otherwise a sample photo."""
    if USE_WEBCAM and IN_COLAB:
        try:
            print(f"Take a photo for the {kind} demo:")
            return take_photo(f"webcam_{kind}.jpg")
        except Exception as e:
            print(f"Webcam didn't work ({type(e).__name__}), using a sample photo instead.")
    url = SAMPLES[kind]
    return fetch(url, f"sample_{kind}{os.path.splitext(url)[1]}")

def rgb(path):
    return cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)

print("Ready. MediaPipe", mp.__version__, "| Colab" if IN_COLAB else "| local")

## 1. What a landmark is

The **Hand Landmarker** finds each hand and returns **21 numbered points**. Each point has:

- $x$ and $y$: position across the image, from 0 to 1. Multiply by the image width and height to get pixels.
- $z$: rough depth relative to the wrist. Smaller means closer to the camera.

Note that $y$ grows **downward**, like most image coordinates.

In [ ]:
hands = vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=model(HAND_MODEL)), num_hands=2))

HAND_NAMES = ["wrist", "thumb_cmc", "thumb_mcp", "thumb_ip", "thumb_tip",
              "index_mcp", "index_pip", "index_dip", "index_tip",
              "middle_mcp", "middle_pip", "middle_dip", "middle_tip",
              "ring_mcp", "ring_pip", "ring_dip", "ring_tip",
              "pinky_mcp", "pinky_pip", "pinky_dip", "pinky_tip"]

hand_photo = photo("hands")
hand_result = hands.detect(mp.Image.create_from_file(hand_photo))
print(f"Found {len(hand_result.hand_landmarks)} hand(s).\n")

if hand_result.hand_landmarks:
    hand = hand_result.hand_landmarks[0]
    print(f"First hand ({hand_result.handedness[0][0].category_name}), a few of its 21 points:")
    for i in [0, 4, 8, 12, 16, 20]:
        p = hand[i]
        print(f"  {i:>2}  {HAND_NAMES[i]:<11} x={p.x:.3f}  y={p.y:.3f}  z={p.z:+.3f}")

    img = rgb(hand_photo); h, w = img.shape[:2]
    plt.figure(figsize=(7, 7)); plt.imshow(img); plt.axis("off")
    for lms in hand_result.hand_landmarks:
        for c in vision.HandLandmarksConnections.HAND_CONNECTIONS:
            a, b = lms[c.start], lms[c.end]
            plt.plot([a.x * w, b.x * w], [a.y * h, b.y * h], color="white", lw=1.5)
        for i, p in enumerate(lms):
            plt.scatter(p.x * w, p.y * h, s=18, color="#FF8A3D", zorder=3)
            plt.annotate(str(i), (p.x * w, p.y * h), xytext=(4, -4), textcoords="offset points", fontsize=8, color="yellow")
    plt.title("Every point has a number: these ids never change"); plt.show()
else:
    print("No hand found. Try another photo with your hand clearly in view.")

## 2. Finger counter, step by step

**Four fingers:** a finger is *up* when its tip is farther from the wrist than its middle joint. Distances work at any hand angle, unlike “is the tip higher than the joint?”, which breaks when the hand turns sideways.

$$\text{finger up} \iff d(\text{tip}, \text{wrist}) > d(\text{middle joint}, \text{wrist})$$

**Thumb:** it folds *sideways* across the palm, so measure from the pinky base (point 17) instead. It’s out when its tip (4) is farther from 17 than its knuckle (3) is, with a 5% margin so it doesn’t flicker:

$$\text{thumb out} \iff d(p_4, p_{17}) > 1.05 \cdot d(p_3, p_{17})$$

In [ ]:
def dist(a, b):
    return math.hypot(a.x - b.x, a.y - b.y)

FINGERS = {"index": (8, 6), "middle": (12, 10), "ring": (16, 14), "pinky": (20, 18)}

def count_fingers(hand, verbose=False):
    up = 0
    for name, (tip, joint) in FINGERS.items():
        t, j = dist(hand[tip], hand[0]), dist(hand[joint], hand[0])
        up += t > j
        if verbose:
            print(f"  {name:<7} tip→wrist {t:.3f}  joint→wrist {j:.3f}  →  {'UP' if t > j else 'down'}")
    t, k = dist(hand[4], hand[17]), dist(hand[3], hand[17])
    thumb = t > 1.05 * k
    if verbose:
        print(f"  {'thumb':<7} tip→pinky {t:.3f}  knuckle→pinky {k:.3f}  →  {'OUT' if thumb else 'tucked'}")
    return up + thumb

counts = []
for n, hand in enumerate(hand_result.hand_landmarks, 1):
    print(f"Hand {n}:")
    counts.append(count_fingers(hand, verbose=True))
    print(f"  = {counts[-1]} finger(s)\n")
if counts:
    print("Total:", " + ".join(map(str, counts)), "=", sum(counts))

## 3. Hands in 3D

Besides image coordinates, MediaPipe returns **world landmarks**: real 3D positions in meters, centered on the hand. Drag the plot around in Jupyter, or change `view_init` to rotate it.

In [ ]:
if hand_result.hand_world_landmarks:
    W = np.array([[p.x, p.y, p.z] for p in hand_result.hand_world_landmarks[0]]) * 100   # meters → cm
    fig = plt.figure(figsize=(7, 6)); ax = fig.add_subplot(projection="3d")
    for c in vision.HandLandmarksConnections.HAND_CONNECTIONS:
        ax.plot(*zip(W[c.start], W[c.end]), color="#2F6FB5", lw=2)
    ax.scatter(W[:, 0], W[:, 1], W[:, 2], color="#FF8A3D", s=30, depthshade=False)
    ax.scatter(*W[[4, 8, 12, 16, 20]].T, color="#E2304A", s=60, depthshade=False, label="fingertips")
    ax.set_xlabel("x (cm)"); ax.set_ylabel("y (cm)"); ax.set_zlabel("z (cm)")
    ax.invert_yaxis(); ax.view_init(elev=-70, azim=-90); ax.legend()
    span = np.ptp(W, axis=0).max()
    print(f"This hand is about {span:.0f} cm across, measured from one photo.")
    plt.title("Hand world landmarks"); plt.show()

## 4. Pose angles

The **Pose Landmarker** returns **33 body points**: 11/12 are shoulders, 13/14 elbows, 15/16 wrists, 23/24 hips, 25/26 knees and 27/28 ankles.

The angle at a joint $B$, between points $A$ and $C$ (say hip, knee, ankle), comes from the dot product:

$$\theta = \arccos\left(\frac{\vec{BA}\cdot\vec{BC}}{\lVert\vec{BA}\rVert\,\lVert\vec{BC}\rVert}\right)$$

A straight leg is about $180^\circ$. A deep squat is under $100^\circ$.

In [ ]:
pose = vision.PoseLandmarker.create_from_options(vision.PoseLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=model(POSE_MODEL)), num_poses=1))

def angle(a, b, c):
    """Angle at b (degrees) between points a-b-c."""
    ba, bc = np.array([a.x - b.x, a.y - b.y]), np.array([c.x - b.x, c.y - b.y])
    cos = ba @ bc / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-9)
    return math.degrees(math.acos(np.clip(cos, -1, 1)))

JOINTS = {"left elbow": (11, 13, 15), "right elbow": (12, 14, 16), "left knee": (23, 25, 27), "right knee": (24, 26, 28)}

pose_photo = photo("pose")
pose_result = pose.detect(mp.Image.create_from_file(pose_photo))
if not pose_result.pose_landmarks:
    print("No body found. Use a photo where the whole body is visible.")
else:
    P = pose_result.pose_landmarks[0]
    img = rgb(pose_photo); h, w = img.shape[:2]
    plt.figure(figsize=(9, 6)); plt.imshow(img); plt.axis("off")
    for c in vision.PoseLandmarksConnections.POSE_LANDMARKS:
        a, b = P[c.start], P[c.end]
        plt.plot([a.x * w, b.x * w], [a.y * h, b.y * h], color="white", lw=1.5)
    for name, (a, b, c) in JOINTS.items():
        deg = angle(P[a], P[b], P[c])
        print(f"{name:<12} {deg:5.0f}°")
        plt.scatter(P[b].x * w, P[b].y * h, s=60, color="#FF8A3D", zorder=3)
        plt.annotate(f"{deg:.0f}°", (P[b].x * w, P[b].y * h), xytext=(8, -8), textcoords="offset points",
                     fontsize=12, color="white", weight="bold", bbox=dict(fc="#11151C", ec="none", alpha=.75))
    plt.title("Joint angles from 3 landmarks each"); plt.show()

**From angles to a rep counter.** A squat counter only needs two states:

1. When the knee angle drops below $100^\circ$, remember “I’m down”.
2. When it rises back above $160^\circ$ while you’re down, count one rep and reset.

The gap between the two thresholds stops tiny wobbles from counting as reps. Here it is on a simulated knee angle:

In [ ]:
t = np.linspace(0, 10, 300)
knee = 135 + 45 * np.cos(2 * np.pi * t / 2.5) + np.random.default_rng(0).normal(0, 4, t.size)   # 4 squats + noise

reps, down, marks = 0, False, []
for i, a in enumerate(knee):
    if a < 100:
        down = True
    if down and a > 160:
        reps += 1; down = False; marks.append(i)

plt.figure(figsize=(9, 3))
plt.plot(t, knee, color="#2F6FB5"); plt.axhline(100, ls="--", color="#E2304A"); plt.axhline(160, ls="--", color="#2DB36A")
plt.scatter(t[marks], knee[marks], color="#FF8A3D", zorder=3, s=60, label="rep counted")
plt.ylabel("knee angle (°)"); plt.xlabel("seconds"); plt.legend(); plt.title(f"{reps} reps counted"); plt.show()

## 5. Mood detector

The **Face Landmarker** returns 478 points and 52 **blendshapes**: scores from 0 to 1 for movements like `mouthSmileLeft`, `browDownLeft`, `eyeBlinkRight` or `jawOpen`. The demo page combines a few of them into moods:

- **Smile:** the average of `mouthSmileLeft` and `mouthSmileRight`.
- **Frown:** the stronger of the mouth-frown scores ($\times 1.8$) and the brow-down scores ($\times 1.1$).
- **Surprise:** raised brows plus an open jaw or wide eyes.

In [ ]:
faces = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=model(FACE_MODEL)), output_face_blendshapes=True, num_faces=1))

def read_mood(cats):
    s = {c.category_name: c.score for c in cats}
    avg = lambda a, b: (s[a] + s[b]) / 2
    mood = {
        "smile": avg("mouthSmileLeft", "mouthSmileRight"),
        "frown": min(1, max(avg("mouthFrownLeft", "mouthFrownRight") * 1.8, avg("browDownLeft", "browDownRight") * 1.1)),
        "surprise": min(1, 0.55 * max(s["browInnerUp"], avg("browOuterUpLeft", "browOuterUpRight"))
                          + 0.45 * max(s["jawOpen"], avg("eyeWideLeft", "eyeWideRight"))),
        "mouth open": s["jawOpen"],
    }
    if mood["smile"] > 0.4:
        mood["frown"] *= 0.4          # a big smile pulls the brows down too
    return mood

face_photo = photo("face")
face_result = faces.detect(mp.Image.create_from_file(face_photo))
if not face_result.face_blendshapes:
    print("No face found. Try another photo.")
else:
    cats = face_result.face_blendshapes[0]
    top = sorted((c for c in cats if c.category_name != "_neutral"), key=lambda c: -c.score)[:8]
    mood = read_mood(cats)
    limits = {"smile": ("Smiling", .45), "frown": ("Frowning", .42), "surprise": ("Surprised", .42)}
    hits = [(v, limits[k][0]) for k, v in mood.items() if k in limits and v > limits[k][1]]
    headline = max(hits)[1] if hits else "Neutral"

    fig, ax = plt.subplots(1, 3, figsize=(15, 4.8), gridspec_kw={"width_ratios": [1, 1.2, 1]})
    ax[0].imshow(rgb(face_photo)); ax[0].axis("off"); ax[0].set_title(headline, fontsize=18)
    ax[1].barh([c.category_name for c in top][::-1], [c.score for c in top][::-1], color="#2F6FB5")
    ax[1].set_xlim(0, 1); ax[1].set_title("Top 8 of 52 raw blendshapes")
    ax[2].barh(list(mood)[::-1], list(mood.values())[::-1], color="#FF8A3D")
    ax[2].set_xlim(0, 1); ax[2].set_title("Combined into moods")
    plt.tight_layout(); plt.show()

## 6. Train your own gesture classifier

Rules like “tip farther than joint” only go so far. For gestures that are hard to describe, like sign-language letters or a custom “secret handshake”, **let a model learn the rule from examples.**

The trick: don’t train on photos. Train on **landmarks**. Each hand becomes 63 numbers (21 points × $x, y, z$), so even a tiny model learns fast from a few hundred examples, with no GPU needed.

We’ll use Google’s free **rock-paper-scissors** dataset: 500 photos in 4 folders (`rock`, `paper`, `scissors`, `none`).

In [ ]:
DATA_URL = "https://storage.googleapis.com/mediapipe-tasks/gesture_recognizer/rps_data_sample.zip"
DATA_DIR = "rps_data_sample"       # later: point this at your own folder of labeled photos

if not os.path.isdir(DATA_DIR):
    zipfile.ZipFile(fetch(DATA_URL, "rps_data_sample.zip")).extractall(".")

LABELS = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
for label in LABELS:
    print(f"{label:<9} {len(os.listdir(os.path.join(DATA_DIR, label)))} photos")

fig, axes = plt.subplots(1, len(LABELS), figsize=(3 * len(LABELS), 3))
for ax, label in zip(axes, LABELS):
    f = sorted(os.listdir(os.path.join(DATA_DIR, label)))[0]
    ax.imshow(rgb(os.path.join(DATA_DIR, label, f))); ax.set_title(label); ax.axis("off")
plt.show()

### Step 1: photos → features

For each photo: find the hand, then **normalize** its landmarks so the model only sees the *shape* of the hand, not where it is or how big:

$$p'_i = \frac{p_i - p_0}{\lVert p_9 - p_0 \rVert}$$

That is, subtract the wrist ($p_0$) and divide by the wrist-to-middle-knuckle distance. Left hands are mirrored ($x \to -x$) so both hands look alike to the model. Photos where no hand is found are skipped.

In [ ]:
detector = vision.HandLandmarker.create_from_options(vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=model(HAND_MODEL)), num_hands=1,
    min_hand_detection_confidence=0.3))

def features(lms, handedness="Right"):
    """21 landmarks → 63 normalized numbers."""
    P = np.array([[p.x, p.y, p.z] for p in lms])
    P = P - P[0]                                  # wrist at the origin
    P = P / (np.linalg.norm(P[9, :2]) + 1e-9)     # scale by wrist → middle knuckle
    if handedness == "Left":
        P[:, 0] = -P[:, 0]                        # mirror left hands
    return P.flatten()

def features_from_image(path):
    r = detector.detect(mp.Image.create_from_file(path))
    if not r.hand_landmarks:
        return None
    return features(r.hand_landmarks[0], r.handedness[0][0].category_name)

X, y, paths, skipped = [], [], [], 0
start = time.time()
for label in LABELS:
    for f in sorted(os.listdir(os.path.join(DATA_DIR, label))):
        path = os.path.join(DATA_DIR, label, f)
        v = features_from_image(path)
        if v is None:
            skipped += 1
            continue
        X.append(v); y.append(label); paths.append(path)
    print(f"{label:<9} done")
X, y = np.array(X), np.array(y)
print(f"\n{len(X)} hands → {X.shape[1]} features each, {skipped} photos skipped (no hand found), {time.time() - start:.0f}s")

### Step 2: train and test

Hold back 20% of the hands for testing, so the score reflects hands the model has never seen. We try two simple models:

- **k-nearest neighbors:** “which training hands is this closest to?”
- **A small neural network** (MLP) with one hidden layer.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

X_train, X_test, y_train, y_test, p_train, p_test = train_test_split(
    X, y, paths, test_size=0.2, stratify=y, random_state=42)

models = {
    "k-nearest neighbors": KNeighborsClassifier(n_neighbors=5),
    "neural network (MLP)": MLPClassifier(hidden_layer_sizes=(64,), max_iter=2000, random_state=0),
}
for name, m in models.items():
    t0 = time.time(); m.fit(X_train, y_train)
    print(f"{name:<22} test accuracy {accuracy_score(y_test, m.predict(X_test)):.0%}   (trained in {time.time() - t0:.2f}s)")

clf = max(models.values(), key=lambda m: accuracy_score(y_test, m.predict(X_test)))
ConfusionMatrixDisplay.from_predictions(y_test, clf.predict(X_test), cmap="Oranges", colorbar=False)
plt.title("Rows: true gesture · Columns: predicted"); plt.show()

### Step 3: see it on test photos

Random photos the model never trained on. Green means correct, red means wrong.

In [ ]:
random.seed(1)
picks = random.sample(range(len(X_test)), 8)
fig, axes = plt.subplots(2, 4, figsize=(13, 6.5))
for ax, i in zip(axes.flat, picks):
    probs = clf.predict_proba([X_test[i]])[0]
    guess = clf.classes_[probs.argmax()]
    ax.imshow(rgb(p_test[i])); ax.axis("off")
    ax.set_title(f"{guess} ({probs.max():.0%})", color="#2DB36A" if guess == y_test[i] else "#E2304A", fontsize=13)
plt.tight_layout(); plt.show()

### Step 4: try it on your own hand

In Colab, make a rock, paper or scissors in front of your webcam and press **Take photo**. Elsewhere, this uses a test photo.

In [ ]:
if USE_WEBCAM and IN_COLAB:
    test_photo = take_photo("my_gesture.jpg")
else:
    test_photo = p_test[0]

v = features_from_image(test_photo)
plt.figure(figsize=(5, 4)); plt.imshow(rgb(test_photo)); plt.axis("off")
if v is None:
    plt.title("No hand found. Try again with your hand clearly in view.")
else:
    probs = clf.predict_proba([v])[0]
    plt.title(f"{clf.classes_[probs.argmax()]} ({probs.max():.0%})", fontsize=16)
    print("  ".join(f"{c}: {p:.0%}" for c, p in zip(clf.classes_, probs)))
plt.show()

### Step 5: use your own gestures

Make a folder with one subfolder per gesture, about 30+ photos each, from a few people, angles and backgrounds:

```
my_gestures/
  thumbs_up/   *.jpg
  peace/       *.jpg
  none/        *.jpg      ← hands doing nothing in particular
```

Then set `DATA_DIR = "my_gestures"` in the dataset cell and run Steps 1–4 again. In Colab, the next cell collects photos with your webcam for you.

In [ ]:
def collect(label, n=20, folder="my_gestures"):
    """Colab: take n webcam photos for one gesture."""
    os.makedirs(os.path.join(folder, label), exist_ok=True)
    for i in range(n):
        print(f"{label}: photo {i + 1} of {n}. Move your hand a little each time.")
        take_photo(os.path.join(folder, label, f"{int(time.time() * 1000)}.jpg"))
    print(f"Saved {n} photos to {folder}/{label}")

# Example (Colab only), then set DATA_DIR = "my_gestures" and re-run Steps 1-4:
# collect("thumbs_up"); collect("peace"); collect("none")

Save the trained model to reuse it later:

In [ ]:
import joblib
joblib.dump({"model": clf, "labels": list(clf.classes_)}, "gesture_classifier.joblib")
print("Saved gesture_classifier.joblib. Load it with joblib.load(...)['model'].predict([features(...)])")

## Build your own

Everything in the live demo page uses the ideas above. Open `index.html` in a text editor to find them:

| In this notebook | In `index.html` | Hackathon idea |
|---|---|---|
| Finger counter | `countHand()` | Rock-paper-scissors against the computer |
| Pose angles | `bodyFrame()`, `drawRobot()` | Workout form coach with a rep counter |
| Mood detector | `readMood()` | Focus monitor that notices yawns and frowns |
| Gesture classifier | (not in the page yet) | Sign-alphabet translator, touchless slide control |

**Stretch goal:** export the training features to JSON and do the nearest-neighbor lookup in JavaScript inside `index.html`. Then your own gestures run live in the browser.

Docs: [MediaPipe guides](https://ai.google.dev/edge/mediapipe/solutions/guide) · [MediaPipe Studio](https://mediapipe-studio.webapps.google.com/home) · Made by ml@psu